In [6]:
import sys
import os
sys.path.append(os.path.abspath('../../'))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from functions.preproc import *
from functions.ensembles import *

from sklearn.model_selection import train_test_split
from sksurv.ensemble import GradientBoostingSurvivalAnalysis
from sksurv.metrics import concordance_index_censored
from sksurv.util import Surv
from sksurv.metrics import integrated_brier_score
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import StratifiedGroupKFold

In [7]:
dataset = pd.read_csv("data_treated.csv")

categorical_variables = []
with open("categorical.txt", "r") as file:
    for line in file:
        categorical_variables.append(line.strip())

numerical_variables = []
with open("numerical.txt", "r") as file:
    for line in file:
        numerical_variables.append(line.strip())

survival_dataset = dataset.drop('Unnamed: 0', axis=1)

median_imputer(survival_dataset, numerical_variables)

feature_columns = categorical_variables + numerical_variables

In [8]:
# Prepare data
data_x = survival_dataset[feature_columns]
data_y = Surv.from_arrays(
    event=survival_dataset['Regional LN recurrence'],
    time=survival_dataset['Survival days']
)
groups = survival_dataset['PERSON_ID']
event_indicator = survival_dataset['Regional LN recurrence']

In [9]:
param_grid = {
    "learning_rate": [0.01, 0.1, 0.2],
    "n_estimators": [100, 200],
    "max_depth": [1, 3, 5],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 5]
}

sgkf = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=12)
cv = sgkf.split(X=data_x, y=event_indicator, groups=groups)
gbm = GradientBoostingSurvivalAnalysis(random_state=12)

In [10]:
from sksurv.metrics import concordance_index_censored
import numpy as np
from sksurv.util import Surv
from sksurv.metrics import integrated_brier_score
import pandas as pd

def score_survival_model(model, X, y):
    prediction = model.predict(X)
    result = concordance_index_censored(y["event"], y["time"], prediction)
    return result[0]

def one_split(Xtrain, ytrain, Xval, yval, model):
    ytrain_struct = Surv.from_arrays(ytrain['event'], ytrain['time'])
    yval_struct = Surv.from_arrays(yval['event'], yval['time'])

    model.fit(Xtrain, ytrain)

    predicted_risk_scores = model.predict(Xval)
    c_index = concordance_index_censored(yval['event'], yval['time'], predicted_risk_scores)[0]

    surv_funcs = model.predict_survival_function(Xval)

    times_percentile = np.percentile(ytrain["time"], np.linspace(10, 90, 100))
    times_all = np.linspace(
        max(np.min(ytrain["time"]), np.min(yval["time"])) + 1e-6,
        min(np.max(ytrain["time"]), np.max(yval["time"])) - 1e-6,
        100
    )
    times_fixed = np.array([2 * 365, 5 * 365, 10 * 365])

    def predict_probs(times):
        return np.asarray([[fn(t) for t in times] for fn in surv_funcs])

    brier_percentile = integrated_brier_score(ytrain_struct, yval_struct, predict_probs(times_percentile), times_percentile)
    brier_all = integrated_brier_score(ytrain_struct, yval_struct, predict_probs(times_all), times_all)
    brier_fixed = integrated_brier_score(ytrain_struct, yval_struct, predict_probs(times_fixed), times_fixed)

    return c_index, brier_percentile, brier_all, brier_fixed
def train_evaluate(X, y, groups, sgkf, model):
    results = []

    for fold_idx, (train_idx, val_idx) in enumerate(sgkf.split(X, y['event'], groups)):
        Xtrain = X.iloc[train_idx]
        ytrain = y[train_idx]
        Xval = X.iloc[val_idx]
        yval = y[val_idx]

        c_idx, brier_pct, brier_all, brier_fix = one_split(Xtrain.values, ytrain, Xval.values, yval, model)

        fold_results = {
            'fold': fold_idx,
            'c_index': c_idx,
            'brier_percentile': brier_pct,
            'brier_alltimes': brier_all,
            'brier_fixed': brier_fix
        }

        results.append(fold_results)

    df = pd.DataFrame(results)

    # Add summary row
    summary = {
        'fold': 'mean',
        'c_index': df['c_index'].mean(),
        'brier_percentile': df['brier_percentile'].mean(),
        'brier_alltimes': df['brier_alltimes'].mean(),
        'brier_fixed': df['brier_fixed'].mean()
    }
    stds = {
        'fold': 'std',
        'c_index': df['c_index'].std(),
        'brier_percentile': df['brier_percentile'].std(),
        'brier_alltimes': df['brier_alltimes'].std(),
        'brier_fixed': df['brier_fixed'].std()
    }

    df = pd.concat([df, pd.DataFrame([summary, stds])], ignore_index=True)

    return df

In [11]:
grid_search = GridSearchCV(
    estimator=gbm,
    param_grid=param_grid,
    cv=cv,
    scoring=score_survival_model,
    n_jobs=-1,
    verbose=1
)

In [12]:
grid_search.fit(data_x, data_y)

Fitting 4 folds for each of 72 candidates, totalling 288 fits


GridSearchCV(cv=<generator object _BaseKFold.split at 0x000001536BE2EC20>,
             estimator=GradientBoostingSurvivalAnalysis(random_state=12),
             n_jobs=-1,
             param_grid={'learning_rate': [0.01, 0.1, 0.2],
                         'max_depth': [1, 3, 5], 'min_samples_leaf': [1, 5],
                         'min_samples_split': [2, 5],
                         'n_estimators': [100, 200]},
             scoring=<function score_survival_model at 0x000001536D203BA0>,
             verbose=1)

In [14]:
print("Best parameters:", grid_search.best_params_)
print("Best C-Index:", grid_search.best_score_)
best_model = grid_search.best_estimator_

Best parameters: {'learning_rate': 0.2, 'max_depth': 3, 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 100}
Best C-Index: 0.7700006795711529


In [ ]:
sgkf = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=12)

'''Best parameters: {'learning_rate': 0.2, 'max_depth': 3, 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 100}
Best C-Index: 0.7700006795711529'''

In [17]:
results = train_evaluate(data_x, data_y, groups, sgkf, best_model)
results

,fold,c_index,brier_percentile,brier_alltimes,brier_fixed
0,0,0.700952,0.013650,0.078245,0.026066
1,1,0.733478,0.013430,0.039893,0.025090
2,2,0.778537,0.014603,0.059791,0.016242
3,3,0.867035,0.004601,0.161336,0.009173
4,mean,0.770001,0.011571,0.084816,0.019143
5,std,0.072088,0.004674,0.053363,0.007981
